In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas
import sklearn
import xgboost
import sklearn.linear_model
import random
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from datetime import datetime
from scipy.io.arff import loadarff

1. Body signal of smoking (XGB Model)

https://www.kaggle.com/datasets/kukuroo3/body-signal-of-smoking

license: CC0: Public Domain

In [ ]:
smoking = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/smoking/smoking.csv")
smoking = smoking.drop(columns = "oral")
smoking

In [ ]:
#hard coding
smoking["gender"] = smoking["gender"].map({"M":1, "F":0})
smoking["tartar"] = smoking["tartar"].map({"Y":1, "N":0})

#standardization
x = smoking[smoking.columns[1:-1]]
y = smoking["smoking"]
ss = StandardScaler()
x = ss.fit_transform(x)

#data split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, print evaluation results
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                             sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                                    max(tpr-fpr)))

2. Heart Attack Analysis & Prediction Dataset (LR Model)

https://www.kaggle.com/datasets/rashikrahmanpritom/heart-attack-analysis-prediction-dataset

license: CC0: Public Domain

In [ ]:
heart = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/heart/heart.csv")
heart

In [ ]:
#woe coding
def woe(data):
    import math
    large1, large0 = sum(data[data.columns[-1]]), sum(data[data.columns[-1]] == False)
    for column in data.columns[:-1]:
        local = {}
        for unique in set(data[column].unique()):
            local_1, local_0 = sum(data[data[column] == unique][data.columns[-1]]), sum(data[data[column] == unique][data.columns[-1]] == False)
            if local_1 == 0:
                local[unique] = 0
            elif local_0 != 0:
                local[unique] = (
                                    math.log(
                                            (local_1/large1)/(local_0/large0)  
                                            )
                                )
            else:
                local[unique] = (
                                (local_1/large1)
                                )
        temp = []
        for row in data[column]:
            temp.append(local[row])
        data[column] = temp
    return data

num = [i for i in heart.columns if len(heart[i].unique()) > 5]

processed = woe(heart[[i for i in heart if i not in num]])
heart[[i for i in heart.columns if i not in num]] = processed

#starnardization
heart[num] = heart[num].apply(lambda x:(x-x.mean())/x.std())

#data split
x = heart[heart.columns[:-1]]
y = heart[heart.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#prediction, print evaluation results
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))

3. Water Quality (LR Model)

https://www.kaggle.com/datasets/adityakadiwal/water-potability

license: CC0: Public Domain

In [ ]:
water = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/water/water_potability.csv")
water

In [ ]:
#filling missing value
for i in water.columns:
    if sum(water[i].isnull()) > 0:
        water[i] = water[i].fillna(water[i].mean())

#standardization
x = water[water.columns[:-1]]
y = water[water.columns[-1]]
x = x.apply(lambda x:(x - x.mean())/x.std())

#data split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#prediction, print evaluation results
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))

4. Customer Personality Analysis (LR Model)

https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis

license: CC0: Public Domain

In [ ]:
customer = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/customer/marketing_campaign.csv", sep="\t")
customer = customer.drop(columns = ["Z_CostContact", "Z_Revenue", "Dt_Customer"])
customer

In [ ]:
#woe coding
processed = woe(customer[["Education", "Marital_Status", "Response"]])
customer[["Education", "Marital_Status", "Response"]] = processed

#filling missing value
customer["Income"] = customer["Income"].fillna(customer["Income"].mean())

#standardization
numf = [i for i in customer.columns if i not in ["ID", "Education", "Marital_Status", "Response"]]
standardize = customer[numf]
ss = StandardScaler()
standardize = ss.fit_transform(standardize)
customer[numf] = standardize

#data split
x = customer[customer.columns[1:-1]]
y = customer[customer.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#prediction, print evaluation results
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))

5. Travel Insurance Prediction Data (XGB Model)

https://www.kaggle.com/datasets/tejashvi14/travel-insurance-prediction-data

license: CC0: Public Domain

In [ ]:
insurance = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/insurance/TravelInsurancePrediction.csv")
insurance

In [ ]:
#binning
num = ["Age", "FamilyMembers"]
bins = KBinsDiscretizer(n_bins = 4, strategy = "uniform", encode = "ordinal")
bins = bins.fit_transform(insurance[num])
insurance[num] = bins

#woe coding
to_woe = [i for i in insurance.columns if i not in ["Unnamed: 0","AnnualIncome"]]
processed = woe(insurance[to_woe])
insurance[to_woe] = processed

#standardization
insurance["AnnualIncome"] = insurance["AnnualIncome"].apply(lambda x: (x - insurance["AnnualIncome"].mean())/insurance["AnnualIncome"].std())

#data split
x = insurance[insurance.columns[1:-1]]
y = insurance[insurance.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, print evaluation
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                               sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                              max(tpr-fpr)))

6. The Home Equity dataset (XGB Model)

https://www.kaggle.com/datasets/ajay1735/hmeq-data

license: CC0: Public Domain

In [ ]:
credit = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/credit/hmeq.csv")
credit

In [ ]:
#filling missing value
credit["REASON"] = credit["REASON"].fillna(credit["REASON"].mode()[0])
credit["JOB"] = credit["JOB"].fillna(credit["JOB"].mode()[0])
for i in ["MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO"]:
    credit[i] = credit[i].fillna(credit[i].mean())
credit["DEBTINC"] = credit["DEBTINC"].apply(lambda x:x if x > 0 else -1)

#binning
num = ["LOAN", "MORTDUE", "VALUE", "YOJ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]
bins = KBinsDiscretizer(n_bins = 4, strategy = "quantile", encode = "ordinal")
bins = bins.fit_transform(credit[num])
credit[num] = bins

num = ["DEROG"]
bins = KBinsDiscretizer(n_bins = 4, strategy = "uniform", encode = "ordinal")
bins = bins.fit_transform(credit[num])
credit[num] = bins

#woe coding
to_woe = [i for i in credit.columns[1:] if i != "DELINQ"] + ["BAD"]
processed = woe(credit[to_woe])
credit[to_woe] = processed

#standardization
credit["DELINQ"] = credit["DELINQ"].apply(lambda x: (x - credit["DELINQ"].mean())/credit["DELINQ"].std())

#data split
x = credit[credit.columns[1:]]
y = credit[credit.columns[0]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, print evaluation results
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                               sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                              max(tpr-fpr)))

7. Income Dataset (XGB Model)

https://www.kaggle.com/datasets/mastmustu/income

license: CC0: Public Domain

In [ ]:
income = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/income/train.csv")
income

In [ ]:
#filling missing value
income["workclass"] = income["workclass"].fillna("-1")
income["occupation"] = income["occupation"].fillna("-1")

#hard coding
developed = ["United-States", "Japan", "South", "Portugal", 
 "Italy", "England", "Germany", "Yugoslavia", 
 "Poland", "Greece", "Ireland", "Canada", 
 "Scotland", "Outlying-US(Guam-USVI-etc)", "Taiwan", "France",
 "Hungary", "Hong", "Holand-Netherlands"]
income["native-country"] = income["native-country"].apply(lambda x: 1 if x in developed else 0)

#binning
num = ["age", "hours-per-week"]
bins = KBinsDiscretizer(n_bins = 4, strategy = "uniform", encode = "ordinal")
bins = bins.fit_transform(income[num])
income[num] = bins

num = ["educational-num"]
bins = KBinsDiscretizer(n_bins = 5, strategy = "quantile", encode = "ordinal")
bins = bins.fit_transform(income[num])
income[num] = bins

#woe coding
numf = ["fnlwgt", "capital-gain", "capital-loss"]

processed = woe(income[[i for i in income.columns if i not in numf]])
income[[i for i in income.columns if i not in numf]] = processed

#standardization
income[numf] = income[numf].apply(lambda x: (x-x.mean())/x.std())

#data split
x = income[income.columns[:-1]]
y = income[income.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, print evaluation results
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                               sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                              max(tpr-fpr)))

8. Machine Predictive Maintenance Classification (LR Model)

https://www.kaggle.com/datasets/shivamb/machine-predictive-maintenance-classification

license: CC0: Public Domain

In [ ]:
machine = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/machine/predictive_maintenance.csv")
machine["y"] = machine["Target"]
machine = machine.drop(columns = ["Target", "Failure Type", "Product ID"])
machine

In [ ]:
#binning
num = ["Air temperature","Process temperature", "Rotational speed", "Torque", "Tool wear"]
bins = KBinsDiscretizer(n_bins = 4, strategy = "uniform", encode = "ordinal")
bins = bins.fit_transform(machine[num])
machine[num] = bins

#woe coding
processed = woe(machine[[i for i in machine.columns if i != "UDI"]])
machine[[i for i in machine.columns if i != "UDI"]] = processed

#data split
x = machine[machine.columns[1:-1]]
y = machine[machine.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#prediction, print evaluation results
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))

9. Lumpy Skin Disease Dataset (LR Model)

license: Attribution 4.0 International (CC BY 4.0)

https://www.kaggle.com/datasets/saurabhshahane/lumpy-skin-disease-dataset

In [ ]:
skin = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/skin/Lumpy skin disease data.csv")
skin["param1"] = skin["x"]
skin["param2"] = skin["y"]
skin = skin.drop(columns = ["x", "y", "region", "country", "reportingDate"])
skin["y"] = skin["lumpy"]
skin = skin.drop(columns = "lumpy")
skin

In [ ]:
#woe coding
processed = woe(skin[["dominant_land_cover", "y"]])
skin[["dominant_land_cover", "y"]] = processed

#standardization
skin[[i for i in skin.columns[:-1] if i != "dominant_land_cover"]] = skin[[i for i in skin.columns[:-1] if i != "dominant_land_cover"]].apply(lambda x:(x-x.mean())/x.std())

#data split
x = skin[skin.columns[:-1]]
y = skin[skin.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#prediction, print evaluation results
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))

10. Credit score classification (XGB Model)

https://www.kaggle.com/datasets/parisrohan/credit-score-classification?select=train.csv

license: CC0: Public Domain

In [ ]:
score = pandas.read_csv("/kaggle/input/datasets-for-federated-learning/wefe-default_data/score/train.csv")
score["y"] = score["Credit_Score"].apply(lambda x:1 if x == "Poor" else 0)
score = score.drop(columns = ["Credit_Score", "Customer_ID", "Name", "SSN", "Type_of_Loan", "Month", "Occupation"])
score

In [ ]:
#preprocessing data
def preprocessing_score(score):
    
    def process_missing(x):
        if type(x) == str:
            if "_" in x:
                return x[:x.index("_")] + "0"
            else:
                return x
        else:
            return x

    def get_months(x):
        if type(x) == str:
            year = int(x[:x.index("Y") - 1])
            month = int(x[x.index("d") + 1:x.index("M") - 1])
            return year*12 + month
        else:
            return x
    
    def drop_(x):
        if type(x) == str:
            if "_" in x:
                return x[:-1]
            else:
                return x
        else:
            return x
    
    score["Num_of_Delayed_Payment"] = score["Num_of_Delayed_Payment"].apply(lambda x: process_missing(x))
    score["Num_of_Delayed_Payment"] = score["Num_of_Delayed_Payment"].apply(lambda x:int(x) if type(x) == str else x)
    score["Num_of_Delayed_Payment"] = score["Num_of_Delayed_Payment"].fillna(score["Num_of_Delayed_Payment"].median())    
    score["Credit_History_Age"] = score["Credit_History_Age"].apply(lambda x:get_months(x))
    fill = score[score["Amount_invested_monthly"] != "__10000__"]["Amount_invested_monthly"].apply(lambda x:float(x)).median()
    score["Amount_invested_monthly"] = score["Amount_invested_monthly"].apply(lambda x:fill if x == "__10000__" else x)
    score["Amount_invested_monthly"] = score["Amount_invested_monthly"].apply(lambda x:float(x))
    fill = score[score["Monthly_Balance"] != "__-333333333333333333333333333__"]["Monthly_Balance"].apply(lambda x:float(x)).mean()
    score["Monthly_Balance"] = score["Monthly_Balance"].apply(lambda x: fill if x == "__-333333333333333333333333333__" else x)
    score["Monthly_Balance"] = score["Monthly_Balance"].apply(lambda x:float(x))
    score["Annual_Income"] = score["Annual_Income"].apply(lambda x:drop_(x))
    score["Annual_Income"] = score["Annual_Income"].apply(lambda x:float(x))
    score["Age"] = score["Age"].apply(lambda x:drop_(x))
    score["Age"] = score["Age"].apply(lambda x:int(x))
    score["Age"] = score["Age"].apply(lambda x:score["Age"].median() if x == -500 else x)
    score["Num_of_Loan"] = score["Num_of_Loan"].apply(lambda x:drop_(x))
    score["Num_of_Loan"] = score["Num_of_Loan"].apply(lambda x:int(x))
    score["Outstanding_Debt"] = score["Outstanding_Debt"].apply(lambda x:drop_(x))
    score["Outstanding_Debt"] = score["Outstanding_Debt"].apply(lambda x:float(x))
    fill = score[score["Changed_Credit_Limit"] != "_"]["Changed_Credit_Limit"].apply(lambda x:float(x)).mean()
    score["Changed_Credit_Limit"] = score["Changed_Credit_Limit"].apply(lambda x:fill if x == "_" else x)
    score["Changed_Credit_Limit"] = score["Changed_Credit_Limit"].apply(lambda x:float(x))
    
    return score

score = preprocessing_score(score)

In [ ]:
#filling missing value
score["Monthly_Inhand_Salary"] = score["Monthly_Inhand_Salary"].fillna(score["Monthly_Inhand_Salary"].median())
score["Num_of_Delayed_Payment"] = score["Num_of_Delayed_Payment"].fillna(score["Num_of_Delayed_Payment"].median())
score["Num_Credit_Inquiries"] = score["Num_Credit_Inquiries"].fillna(score["Num_Credit_Inquiries"].median())
score["Credit_History_Age"] = score["Credit_History_Age"].fillna(score["Credit_History_Age"].median())
score["Num_Credit_Inquiries"] = score["Num_Credit_Inquiries"].fillna(score["Num_Credit_Inquiries"].median())
score["Amount_invested_monthly"] = score["Amount_invested_monthly"].fillna(score["Amount_invested_monthly"].median())
score["Monthly_Balance"] = score["Monthly_Balance"].fillna(score["Monthly_Balance"].mean())

#woe coding
to_woe = [i for i in score.columns[1:-1] if len(score[i].unique()) <= 16]
processed = woe(score[to_woe + ["y"]])
score[to_woe + ["y"]] = processed

#standardization
score[[i for i in score.columns[1:-1] if len(score[i].unique()) > 16]] = score[[i for i in score.columns[1:-1] if len(score[i].unique()) > 16]].apply(lambda x:(x-x.mean())/x.std())

#data split
x = score[score.columns[1:-1]]
y = score[score.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, print evaluation results
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                               sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                              max(tpr-fpr)))